# Hiver SDE Intern Assignment - Classical ML + LangGraph HITL Pipeline

This notebook demonstrates a complete customer support pipeline combining:
1. **Data Preparation**: Processing the Kaggle Twitter Customer Support dataset.
2. **Classical ML Front-Door**: Training a TF-IDF + Logistic Regression model to predict intents instantly.
3. **RAG (Retrieval-Augmented Generation)**: Indexing historical company responses.
4. **LangGraph Pipeline with LLM-as-a-Judge**: Using an LLM to evaluate if the RAG context safely resolves the user's issue, or if it requires a Human-in-the-Loop (HITL) escalation (e.g. for account refunds).

In [2]:
!pip install -q pandas scikit-learn langchain-google-genai langchain-community chromadb langgraph sentence-transformers

In [2]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from typing import TypedDict

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langgraph.graph import StateGraph, END

import getpass
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Enter your HF Token: ")
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google Gemini API Key: ")
    
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### 1. Dataset Preparation (Kaggle twcs.csv format)

# 01 — TWCS preprocessing and brand selection

**Goal:** select one support brand using the largest number of reconstructed conversations, then create a brand-only corpus for both the classifier and RAG.

Primary selection criterion: **unique reconstructed conversations** rather than raw tweet count.

In [3]:
import pandas as pd
import numpy as np
import zipfile
from pathlib import Path

ZIP_PATH = Path(r"data/archive.zip")
RAW_MEMBER = 'twcs/twcs.csv'
USE_COLS = [
    'tweet_id', 'author_id', 'inbound', 'created_at', 'text',
    'response_tweet_id', 'in_response_to_tweet_id'
]


In [4]:
# Load the raw corpus. The compressed CSV is ~516 MB on disk.
with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open(RAW_MEMBER) as f:
        df = pd.read_csv(f, usecols=USE_COLS)

print('Shape:', df.shape)
print(df.columns.tolist())
print(df['inbound'].value_counts(dropna=False))


Shape: (2811774, 7)
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']
inbound
True     1537843
False    1273931
Name: count, dtype: int64


## Reconstruct conversation IDs

Each tweet has at most one `in_response_to_tweet_id`. We use that parent pointer to walk to the root tweet. A vectorized parent-array implementation keeps this fast enough for the full corpus.

In [5]:
tweet_ids = df['tweet_id'].to_numpy(dtype=np.int64)
max_id = int(tweet_ids.max())
parent_by_id = np.zeros(max_id + 1, dtype=np.int64)
parents = df['in_response_to_tweet_id'].fillna(0).astype(np.int64).to_numpy()
parent_by_id[tweet_ids] = parents

roots = tweet_ids.copy()
for _ in range(100):
    next_roots = parent_by_id[roots]
    next_roots = np.where(next_roots != 0, next_roots, roots)
    if np.array_equal(next_roots, roots):
        break
    roots = next_roots

df['conversation_id'] = roots
print('Unique conversations:', df['conversation_id'].nunique())


Unique conversations: 799263


## Rank support brands

`inbound=False` rows are support/company messages. Their `author_id` is the support account. We count the unique reconstructed conversations in which each support account participates.

In [6]:
support = df.loc[~df['inbound'], ['author_id', 'conversation_id']]
brand_stats = (
    support.groupby('author_id')
    .agg(
        support_tweets=('conversation_id', 'size'),
        conversations=('conversation_id', 'nunique')
    )
    .sort_values(['conversations', 'support_tweets'], ascending=False)
)

brand_stats.head(20)


,support_tweets,conversations
author_id,,
AmazonHelp,169840,82534
AppleSupport,106860,80702
Uber_Support,56270,41923
SpotifyCares,43265,28280
AmericanAir,36764,26385
Delta,42253,26166
comcastcares,33031,24061
TMobileHelp,34317,22789
SouthwestAir,28977,21636


In [7]:
SELECTED_BRAND = brand_stats.index[0]
print('Selected brand:', SELECTED_BRAND)
print('Reconstructed conversations:', brand_stats.loc[SELECTED_BRAND, 'conversations'])


Selected brand: AmazonHelp
Reconstructed conversations: 82534


## Slice to the selected brand

We define the brand slice as conversations in which the selected support account replied. We retain customer messages in those conversations and support messages authored by the selected brand. Other support accounts appearing in the same root conversation are excluded from the brand corpus so their responses do not enter the RAG knowledge base.

In [8]:
brand_conversation_ids = set(
    df.loc[(~df['inbound']) & (df['author_id'] == SELECTED_BRAND), 'conversation_id'].unique()
)

brand_df = df[df['conversation_id'].isin(brand_conversation_ids)].copy()
brand_df = brand_df[(brand_df['inbound']) | (brand_df['author_id'] == SELECTED_BRAND)].copy()
brand_df['message_role'] = np.where(brand_df['inbound'], 'customer', 'support')

print('Brand:', SELECTED_BRAND)
print('Conversations:', brand_df['conversation_id'].nunique())
print('Customer messages:', brand_df['inbound'].sum())
print('Support messages:', (~brand_df['inbound']).sum())


Brand: AmazonHelp
Conversations: 82534
Customer messages: 203251
Support messages: 169840


In [9]:
classifier_df = brand_df.loc[brand_df['inbound'], [
    'conversation_id', 'tweet_id', 'created_at', 'text', 'in_response_to_tweet_id'
]].copy()

rag_df = brand_df.loc[~brand_df['inbound'], [
    'conversation_id', 'tweet_id', 'created_at', 'text',
    'in_response_to_tweet_id', 'response_tweet_id'
]].copy()

print('Classifier rows:', len(classifier_df))
print('RAG/support rows:', len(rag_df))


Classifier rows: 203251
RAG/support rows: 169840


In [10]:
# Save outputs
out = Path('data/processed')
out.mkdir(parents=True, exist_ok=True)

brand_df.to_csv(out / 'amazonhelp_brand_conversations.csv.gz', index=False, compression='gzip')
classifier_df.to_csv(out / 'amazonhelp_customer_messages.csv.gz', index=False, compression='gzip')
rag_df.to_csv(out / 'amazonhelp_support_messages.csv.gz', index=False, compression='gzip')
brand_stats.to_csv(out / 'brand_conversation_stats.csv')

print('Saved to', out.resolve())


Saved to C:\Users\Sahil Kaushik\Downloads\classical-ml-&-langgraph-hitl-pipeline (2)\data\processed


## Why Only AmazonHelp
AmazonHelp was selected because it had the largest number of reconstructed customer-support conversations in the provided TWCS corpus (82,534), giving us the largest brand-specific pool for intent modeling and historical-response retrieval.

In [12]:
df = pd.read_csv(r'data/processed/amazonhelp_brand_conversations.csv.gz')
inbound = df[df['inbound'] == True].copy()
outbound = df[df['inbound'] == False].copy()

# Merge inbound queries with company responses
merged = pd.merge(inbound, outbound, left_on='tweet_id', right_on='in_response_to_tweet_id', suffixes=('_user', '_company'))
merged['user_text'] = merged['text_user'].str.replace(r'^@\w+\s+', '', regex=True)
merged['company_response'] = merged['text_company'].str.replace(r'^@\w+\s+', '', regex=True)

# Heuristic pseudo-labels for training the classifier
def assign_intent(text):
    text = text.lower()
    if any(w in text for w in ['charge', 'refund', 'fee', 'bill']): return 'BILLING'
    if any(w in text for w in ['track', 'package', 'delivered']): return 'SHIPPING'
    if any(w in text for w in ['crash', 'error', 'reset']): return 'TECHNICAL'
    return 'GENERAL_INQUIRY'

merged['intent'] = merged['user_text'].apply(assign_intent)
merged[['user_text', 'company_response', 'intent']].head()

,user_text,company_response,intent
0,電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎてるので買い直しになるん...,カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました。リプライいただ...,GENERAL_INQUIRY
1,こちらこそありがとうございました。,恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお願いします。ET,GENERAL_INQUIRY
2,amazonのfireTVstickが見れない😢,こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは、どのような状況...,GENERAL_INQUIRY
3,amazonプライムビデオ、再生エラーが多いです,ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の再起動にて改善す...,GENERAL_INQUIRY
4,3 different people have given 3 different answ...,We'd like to take a further look into this wit...,SHIPPING


### 2. Train Classical ML Classifier & Setup RAG

In [ ]:
from tqdm.notebook import tqdm
# Train Classifier
classifier = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
classifier.fit(merged['user_text'], merged['intent'])
print("Classifier trained.")

# Setup RAG Vectorstore
docs = []
for _, row in merged.iterrows():
    docs.append(Document(page_content=row['user_text'], metadata={"company_response": row['company_response'], "intent": row['intent']}))

vectorstore = Chroma(embedding_function=embeddings, persist_directory="./chroma_db")
batch_size = 5000

for i in tqdm(range(0, len(docs), batch_size), desc="Generating Embeddings"):
    batch = docs[i : i + batch_size]
    vectorstore.add_documents(batch)

print("All rows embedded and saved successfully")
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [15]:
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

# =========================================================
# 1. TRAIN CLASSIFIER
# =========================================================

# Split the labelled data for validation
X_train, X_val, y_train, y_val = train_test_split(
    merged['user_text'],
    merged['intent'],
    test_size=0.2,
    random_state=42,
    stratify=merged['intent']
)

classifier = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=5000
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced'
    ))
])

classifier.fit(X_train, y_train)

print("Classifier trained.")


# =========================================================
# 2. CLASSIFIER VALIDATION
# =========================================================

val_predictions = classifier.predict(X_val)
val_probabilities = classifier.predict_proba(X_val)

print("\nClassification Validation Results")
print("=" * 50)
print("Accuracy :", accuracy_score(y_val, val_predictions))
print("Macro F1 :", f1_score(y_val, val_predictions, average='macro'))
print("\nClassification Report:")
print(classification_report(y_val, val_predictions))


# =========================================================
# 3. ML OUTPUT VALIDATION
# =========================================================

def validate_ml_output(text, confidence_threshold=0.60, margin_threshold=0.15):
    """
    Returns prediction + uncertainty information.
    This will later be passed to the LLM Judge.
    """

    probabilities = classifier.predict_proba([text])[0]
    classes = classifier.classes_

    # Sort probabilities
    ranked = probabilities.argsort()[::-1]

    top1_idx = ranked[0]
    top2_idx = ranked[1]

    predicted_intent = classes[top1_idx]

    confidence = float(probabilities[top1_idx])
    second_confidence = float(probabilities[top2_idx])

    # Difference between best and second-best class
    margin = confidence - second_confidence

    # Entropy = uncertainty across all classes
    entropy = float(
        -(probabilities * np.log(probabilities + 1e-12)).sum()
    )

    is_uncertain = (
        confidence < confidence_threshold
        or margin < margin_threshold
    )

    return {
        "intent": predicted_intent,
        "confidence": confidence,
        "margin": margin,
        "entropy": entropy,
        "is_uncertain": is_uncertain,
        "probabilities": {
            cls: float(prob)
            for cls, prob in zip(classes, probabilities)
        }
    }

# =========================================================
# 4. SETUP RAG VECTORSTORE
# =========================================================

docs = []

for _, row in merged.iterrows():

    docs.append(
        Document(
            page_content=row['user_text'],
            metadata={
                "company_response": row['company_response'],
                "intent": row['intent']
            }
        )
    )


vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

batch_size = 5000

for i in tqdm(
    range(0, len(docs), batch_size),
    desc="Generating Embeddings"
):
    batch = docs[i:i + batch_size]
    vectorstore.add_documents(batch)

print("All rows embedded and saved successfully.")


# =========================================================
# 5. RAG RETRIEVER
# =========================================================

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


# =========================================================
# 6. RAG RETRIEVAL VALIDATION
# =========================================================

def validate_retrieval(
    query,
    k=3,
    min_relevant_docs=2,
    min_score=0.50
):
    """
    Retrieves top-k documents and checks whether
    enough relevant historical evidence exists.
    """

    results = vectorstore.similarity_search_with_relevance_scores(
        query,
        k=k
    )

    if not results:
        return {
            "retrieval_status": "BAD",
            "relevant_documents": 0,
            "max_score": 0.0,
            "average_score": 0.0,
            "documents": []
        }

    scores = [float(score) for _, score in results]

    relevant_docs = [
        (doc, score)
        for doc, score in results
        if score >= min_score
    ]

    max_score = max(scores)
    average_score = sum(scores) / len(scores)

    retrieval_status = (
        "GOOD"
        if len(relevant_docs) >= min_relevant_docs
        else "BAD"
    )

    return {
        "retrieval_status": retrieval_status,
        "relevant_documents": len(relevant_docs),
        "max_score": max_score,
        "average_score": average_score,
        "documents": [
            {
                "customer_query": doc.page_content,
                "company_response": doc.metadata.get(
                    "company_response", ""
                ),
                "intent": doc.metadata.get(
                    "intent", ""
                ),
                "score": float(score)
            }
            for doc, score in results
        ]
    }


# Example RAG validation
retrieval_result = validate_retrieval(
    "I was charged twice on my card this month."
)

print("\nRAG Retrieval Validation")
print("=" * 50)
print("Status:", retrieval_result["retrieval_status"])
print("Relevant documents:",
      retrieval_result["relevant_documents"])
print("Max score:",
      retrieval_result["max_score"])
print("Average score:",
      retrieval_result["average_score"])

Classifier trained.

Classification Validation Results
Accuracy : 0.9860794360690697
Macro F1 : 0.898514250795826

Classification Report:
                 precision    recall  f1-score   support

        BILLING       0.97      0.93      0.95      2059
GENERAL_INQUIRY       0.99      0.99      0.99     28077
       SHIPPING       0.99      0.98      0.99      3384
      TECHNICAL       0.55      0.86      0.67       243

       accuracy                           0.99     33763
      macro avg       0.87      0.94      0.90     33763
   weighted avg       0.99      0.99      0.99     33763



Generating Embeddings:   0%|          | 0/34 [00:00<?, ?it/s]

All rows embedded and saved successfully.

RAG Retrieval Validation
Status: GOOD
Relevant documents: 3
Max score: 0.7607891549554473
Average score: 0.7607891549554472


### 3. Build LangGraph Pipeline (with LLM-as-a-Judge)

In [16]:
from langchain_core.output_parsers import StrOutputParser
class AgentState(TypedDict):
    query: str
    predicted_intent: str
    retrieved_responses: list[str]
    judge_decision: str
    final_response: str

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)
output_parser = StrOutputParser()

def classify_intent(state: AgentState):
    return {"predicted_intent": classifier.predict([state['query']])[0]}

def retrieve_context(state: AgentState):
    docs = retriever.invoke(state['query'])
    return {"retrieved_responses": [doc.metadata['company_response'] for doc in docs]}

def llm_judge(state: AgentState):
    prompt = f"""You are a routing judge.
    User Query: {state['query']}
    Historical response from knowledge base: {state['retrieved_responses'][0]}
    
    Task: If the historical response contains generic steps/links that solve the issue, output 'RAG_VALID'. 
    If the issue requires checking their account, processing refunds, or requesting personal info (like DMing account numbers), you MUST output 'HITL'."""
    decision = "HITL" if "HITL" in llm.invoke(prompt).content else "RAG_VALID"
    return {"judge_decision": decision}

def route_decision(state: AgentState):
    return state['judge_decision']

def generate_rag_response(state: AgentState):
    prompt = f"Write a polite reply to: '{state['query']}' using this knowledge: {state['retrieved_responses'][0]}"
    chain = llm | output_parser
    response_text = chain.invoke(prompt)
    
    return {"final_response": response_text.strip()}

def hitl_fallback(state: AgentState):
    return {"final_response": "[ESCALATED TO HUMAN QUEUE]: Account verification or refund processing required."}

workflow = StateGraph(AgentState)
workflow.add_node("classify", classify_intent)
workflow.add_node("retrieve", retrieve_context)
workflow.add_node("judge", llm_judge)
workflow.add_node("generate_reply", generate_rag_response)
workflow.add_node("hitl", hitl_fallback)

workflow.set_entry_point("classify")
workflow.add_edge("classify", "retrieve")
workflow.add_edge("retrieve", "judge")
workflow.add_conditional_edges("judge", route_decision, {"RAG_VALID": "generate_reply", "HITL": "hitl"})
workflow.add_edge("generate_reply", END)
workflow.add_edge("hitl", END)

app = workflow.compile()

In [17]:
from typing import TypedDict
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

# ---------------------------------------------------------
# 1. STATE
# ---------------------------------------------------------

class AgentState(TypedDict, total=False):
    query: str
    predicted_intent: str
    retrieved_responses: list[str]

    judge_decision: str
    judge_reason: str

    human_response: str
    final_response: str


# ---------------------------------------------------------
# 2. LLM
# ---------------------------------------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

output_parser = StrOutputParser()


# ---------------------------------------------------------
# 3. CLASSIFIER
# ---------------------------------------------------------

def classify_intent(state: AgentState):
    prediction = classifier.predict([state["query"]])[0]

    return {
        "predicted_intent": prediction
    }


# ---------------------------------------------------------
# 4. RAG RETRIEVAL
# ---------------------------------------------------------

def retrieve_context(state: AgentState):

    docs = retriever.invoke(state["query"])

    return {
        "retrieved_responses": [
            doc.metadata["company_response"]
            for doc in docs
        ]
    }


# ---------------------------------------------------------
# 5. LLM ROUTING JUDGE
# ---------------------------------------------------------
from pydantic import BaseModel, Field
from typing import Literal

class JudgeDecision(BaseModel):
    decision: Literal["RAG_VALID", "HITL"]
    reason: str

judge_llm = llm.with_structured_output(JudgeDecision)

def llm_judge(state: AgentState):

    historical_response = (
        state["retrieved_responses"][0]
        if state["retrieved_responses"]
        else "No historical response available."
    )

    prompt = f"""
You are a customer-support routing judge.

Customer Query:
{state["query"]}

Predicted Intent:
{state["predicted_intent"]}

Historical Support Response:
{historical_response}

Decide whether to use RAG or HITL.

Use HITL for:
- account-specific actions
- refunds or payment disputes
- sensitive information
- ambiguous/high-impact cases
- insufficient historical evidence

Use RAG_VALID only when the request can be safely answered
using the historical support knowledge.
"""

    decision = judge_llm.invoke(prompt)

    return {
        "judge_decision": decision.decision,
        "judge_reason": decision.reason
    }

# ---------------------------------------------------------
# 6. ROUTER
# ---------------------------------------------------------

def route_decision(state: AgentState):
    return state["judge_decision"]


# ---------------------------------------------------------
# 7. RAG RESPONSE GENERATION
# ---------------------------------------------------------

def generate_rag_response(state: AgentState):

    historical_response = (
        state["retrieved_responses"][0]
        if state["retrieved_responses"]
        else ""
    )

    prompt = f"""
Write a polite and concise customer-support reply.

Customer query:
{state["query"]}

Historical AmazonHelp support response:
{historical_response}

Use the historical response as grounding.
Do not invent account-specific information.
Do not claim that an action has been performed.
"""

    chain = llm | output_parser

    response_text = chain.invoke(prompt)

    return {
        "final_response": response_text.strip()
    }


# ---------------------------------------------------------
# 8. HUMAN-IN-THE-LOOP
# ---------------------------------------------------------

def hitl_fallback(state: AgentState):

    # Pause the graph and ask the human for the actual response.
    human_input = interrupt({
        "type": "human_response_required",
        "message": "Please provide the final response to the customer.",
        "customer_query": state["query"],
        "predicted_intent": state["predicted_intent"],
        "judge_reason": state.get("judge_reason", ""),
        "historical_response": (
            state["retrieved_responses"][0]
            if state.get("retrieved_responses")
            else None
        )
    })

    # Graph resumes here after Command(resume=...)
    human_response = human_input["human_response"]

    return {
        "human_response": human_response,
        "final_response": human_response
    }


# ---------------------------------------------------------
# 9. BUILD GRAPH
# ---------------------------------------------------------

workflow = StateGraph(AgentState)

workflow.add_node("classify", classify_intent)
workflow.add_node("retrieve", retrieve_context)
workflow.add_node("judge", llm_judge)
workflow.add_node("generate_reply", generate_rag_response)
workflow.add_node("hitl", hitl_fallback)

workflow.set_entry_point("classify")

workflow.add_edge("classify", "retrieve")
workflow.add_edge("retrieve", "judge")

workflow.add_conditional_edges(
    "judge",
    route_decision,
    {
        "RAG_VALID": "generate_reply",
        "HITL": "hitl"
    }
)

workflow.add_edge("generate_reply", END)
workflow.add_edge("hitl", END)


# ---------------------------------------------------------
# 10. CHECKPOINTER
# ---------------------------------------------------------

checkpointer = InMemorySaver()

app = workflow.compile(
    checkpointer=checkpointer
)

### 4. Execute Pipeline

In [19]:
from langgraph.types import Command

test_queries = [
    "Where is my package? I want to track my delivery.",
    "I was charged twice on my card this month, help!"
]


for i, q in enumerate(test_queries, start=1):

    thread_id = f"test_customer_{i}"

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    print("\n" + "=" * 70)
    print(f"User: {q}")
    print(f"Thread ID: {thread_id}")

    # -------------------------------------------------
    # First invocation
    # -------------------------------------------------

    result = app.invoke(
        {"query": q},
        config=config
    )

    # Check whether the graph paused at an interrupt
    snapshot = app.get_state(config)

    if snapshot.next:
        print("\nGraph paused for HITL.")
        print("Next node:", snapshot.next)

        # The interrupt payload is available in the task information
        interrupt_info = snapshot.tasks[0].interrupts[0]

        print("\nHuman is shown:")
        print(interrupt_info.value)

        # -------------------------------------------------
        # Simulate the human response
        # -------------------------------------------------

        human_response = input(
            "\nEnter the human's final response: "
        )

        # -------------------------------------------------
        # Resume the SAME checkpoint/thread
        # -------------------------------------------------

        result = app.invoke(
            Command(
                resume={
                    "human_response": human_response
                }
            ),
            config=config
        )

    # -------------------------------------------------
    # Final result
    # -------------------------------------------------

    print("\nFinal State")
    print("-" * 40)
    print("Intent:", result.get("predicted_intent"))
    print("Judge Decision:", result.get("judge_decision"))
    print("Final Response:", result.get("final_response"))

    # -------------------------------------------------
    # Show final checkpoint
    # -------------------------------------------------

    final_snapshot = app.get_state(config)

    print("\nCheckpoint State")
    print("-" * 40)
    print("Thread ID:", thread_id)
    print("Next:", final_snapshot.next)


User: Where is my package? I want to track my delivery.
Thread ID: test_customer_1


C:\Users\Sahil Kaushik\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
C:\Users\Sahil Kaushik\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Final State
----------------------------------------
Intent: SHIPPING
Judge Decision: RAG_VALID
Final Response: Hello! We would be happy to help. You can track your order here: https://t.co/eVyTqezdQ0

Checkpoint State
----------------------------------------
Thread ID: test_customer_1
Next: ()

User: I was charged twice on my card this month, help!
Thread ID: test_customer_2


C:\Users\Sahil Kaushik\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Graph paused for HITL.
Next node: ('hitl',)

Human is shown:
{'type': 'human_response_required', 'message': 'Please provide the final response to the customer.', 'customer_query': 'I was charged twice on my card this month, help!', 'predicted_intent': 'BILLING', 'judge_reason': 'The customer query involves a duplicate charge on their card, which constitutes a payment dispute and requires account-specific investigation.', 'historical_response': "I'm sorry! What form of payment did you use (i.e. credit, debit, gift card)? Is the charge pending or posted to your acct? ^SH"}



Enter the human's final response:  Sorry, for that. I will check it with the accounts team and get back to you.



Final State
----------------------------------------
Intent: BILLING
Judge Decision: HITL
Final Response: Sorry, for that. I will check it with the accounts team and get back to you.

Checkpoint State
----------------------------------------
Thread ID: test_customer_2
Next: ()
